# 05 · Pregel 引擎（Actor / Channel / Topic）

**对应章节**：LangGraph 教程第 05 章 —— 揭开底层：Pregel 计算模型、Actor、Channel、Topic。

**演示概念**：
- 直接用 `langgraph.pregel.Pregel` + `NodeBuilder` 搭建图（绕过 StateGraph 的高层封装）；
- `EphemeralValue` / `Topic` 两类 Channel 的语义差异（`Topic(accumulate=True/False)`）；
- 节点如何 `subscribe` 某 channel、`write_to` 某 channel；
- `Send` + `Topic` 实现 fan-out 收集（`graphwithtopic.py`）。

**运行前置**：
- 不需要 API Key（纯结构演示）；
- 需要已 `uv sync`。

比如如下的一个简单的Workflow，包含两个节点node1和node2，

In [9]:
from langgraph.graph import StateGraph,START,END
from pprint import pprint
from typing import TypedDict

class AgentState(TypedDict):
    content: str

def Node1(state: AgentState) -> dict:
    return {'content': 'node1'}

def Node2(state: AgentState) -> dict:
    return {'content': 'node2'}

graph = StateGraph(AgentState)
workflow = graph.add_node("node1",Node1)\
					.add_node("node2",Node2)\
					.add_edge(START,"node1")\
					.add_edge("node1","node2")\
					.add_edge("node2",END).compile()
					
print('####nodes:\n')
pprint(workflow.nodes)

print('####channels:\n')
pprint(workflow.channels)

print('\n####pregelNode:')
for key,value in workflow.nodes.items():
    print(f'node: {key},\n\tchannels: {value.channels}, \n\ttriggers:{value.triggers}, \n\twritters: {value.writers}')

####nodes:

{'__start__': <langgraph.pregel._read.PregelNode object at 0x11a2da050>,
 'node1': <langgraph.pregel._read.PregelNode object at 0x11a4f8650>,
 'node2': <langgraph.pregel._read.PregelNode object at 0x11a4f8950>}
####channels:

{'__pregel_tasks': <langgraph.channels.topic.Topic object at 0x11a4db0c0>,
 '__start__': <langgraph.channels.ephemeral_value.EphemeralValue object at 0x11a4db280>,
 'branch:to:node1': <langgraph.channels.ephemeral_value.EphemeralValue object at 0x11a4db3c0>,
 'branch:to:node2': <langgraph.channels.ephemeral_value.EphemeralValue object at 0x11a4d9e40>,
 'content': <langgraph.channels.last_value.LastValue object at 0x11a4da340>}

####pregelNode:
node: __start__,
	channels: __start__, 
	triggers:['__start__'], 
	writters: [ChannelWrite<...,...>(tags=None, recurse=True, explode_args=False, func_accepts={'config': ('N/A', <class 'inspect._empty'>)}, writes=(ChannelWriteTupleEntry(mapper=<function CompiledStateGraph.attach_node.<locals>._get_updates at 0x11a

## 1) 最小 Pregel：`pregelgraph.py`

不依赖 LLM。两个节点 `node1` / `node2` 操作三个 channel `a` / `b` / `c`，
其中 `c` 是 `Topic(str, accumulate=True)`。运行后打印结果，体会“Channel”如何替代 TypedDict 状态。

In [ ]:
from langgraph.pregel import Pregel, NodeBuilder
from langgraph.channels import EphemeralValue, Topic

def node1_func(x: str) -> str:
    return x + x

def node2_func(x: dict) -> str:
    return x["b"] + x["b"]

# 构建两个节点
node1 = (
    NodeBuilder()
    .subscribe_only("a")
    .do(node1_func)
    .write_to("b", "c")
)

node2 = (
    NodeBuilder()
    .subscribe_to("b")
    .do(node2_func)
    .write_to("c")
)

app = Pregel(
    nodes={"node1": node1, "node2": node2},
    channels={
        "a": EphemeralValue(str),
        "b": EphemeralValue(str),
        "c": Topic(str, accumulate=True),
    },
    input_channels=["a"],
    output_channels=["c"],
)

result = app.invoke({"a": "foo"})
print(result)  # {'c': ['foofoo', 'foofoofoofoo']}

## 2) Topic + Send 的 fan-out：`graphwithtopic.py`

不依赖 LLM。演示用 `Topic` 作为 worker 输出通道，配合 `Send` 把任务分发给多个 `worker` 节点，
再在 `collector` 里用 `state["worker_outputs"]`（一个列表）汇总。

> 注意：本段源码来自 demo 时只写到图构建（`add_conditional_edges`），未包含 `.compile()` / `invoke()``，
> 作为“Topic channel 用法”的结构示例阅读即可；如需跑通可参考上方 `pregelgraph.py` 补上编译与调用。

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.channels import Topic
from typing import Annotated, Sequence,TypedDict
from langgraph.types import Send
from langgraph.checkpoint.memory import MemorySaver

# 状态定义
class State(TypedDict):
    worker_outputs: Annotated[Sequence[str], Topic(str, accumulate=False)]
    final_result: str

# 节点
def router(state: State) -> list[Send]:
    return [Send("worker", {"input": f"task_{i}"}) for i in range(3)]

def worker(state: State) -> dict:
    result = f"processed_{state['input']}"
    # 写入 Topic channel（返回字典，key 是 channel 名）
    return {"worker_outputs": result}

def collector(state: State) -> dict:
    outputs = state["worker_outputs"]  # 列表，包含所有 worker 的输出
    combined = ", ".join(outputs)
    return {"final_result": f"collected: [{combined}]"}

# 构建图
builder = StateGraph(State)
builder.add_node("router", router)
builder.add_node("worker", worker)
builder.add_node("collector", collector)

builder.add_edge(START, "router")
# router 分发到 worker（通过 Send）
builder.add_conditional_edges("router", lambda x: x, path_map=None)

### 小结
- StateGraph 只是 Pregel 的一层友好封装；理解 Channel / Actor 有助于排查复杂并发与状态问题；
- `Topic` 适合“多生产者写入、按顺序累积”的场景（如并行 worker 汇总）；
- `EphemeralValue` 只保留最新值，更像普通 State 字段。